> **Deprecated**
>
> This notebook has been consolidated into `secondary_documentation_notebook.ipynb`.
> Please refer to that notebook instead.
> 
> Section: **Reprojecting Representative Samples** (Section 7 in `secondary_documentation_notebook.ipynb`).

In [3]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import numpy as np
import glob
import os

def convert_to_16bit_heightmaps(
    input_folder="Representative_Samples",
    output_folder="16_bit_heightmaps",
    min_height=-100,
    max_height=10000
):
    """
    Wandelt DEMs in 16-bit Heightmaps um mit orthographischer Projektion.
    
    min_height: Minimale Höhe in Metern (wird zu 0)
    max_height: Maximale Höhe in Metern (wird zu 65535)
    """
    # Ausgabeordner erstellen
    output_path = os.path.join(input_folder, output_folder)
    os.makedirs(output_path, exist_ok=True)
    
    # Alle TIF-Dateien finden
    tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
    
    if not tif_files:
        print(f"Keine TIF-Dateien in {input_folder} gefunden")
        return
    
    print(f"Konvertiere {len(tif_files)} Dateien...")
    
    for tif_file in tif_files:
        filename = os.path.basename(tif_file)
        output_file = os.path.join(output_path, filename)
        
        # Prüfen ob bereits existiert
        if os.path.exists(output_file):
            print(f"Überspringe {filename} (existiert bereits)")
            continue
        
        # DEM öffnen
        with rasterio.open(tif_file) as src:
            # Mittelpunkt des DEMs berechnen
            bounds = src.bounds
            center_lon = (bounds.left + bounds.right) / 2
            center_lat = (bounds.bottom + bounds.top) / 2
            
            # Orthographische Projektion um Mittelpunkt definieren
            ortho_crs = CRS.from_proj4(
                f"+proj=ortho +lat_0={center_lat} +lon_0={center_lon} "
                f"+x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs"
            )
            
            # Transform und Dimensionen für Reprojection berechnen
            transform, width, height = calculate_default_transform(
                src.crs, ortho_crs, src.width, src.height, *bounds
            )
            
            # Neues Array für reprojizierte Daten
            dem_reprojected = np.empty((height, width), dtype=np.float32)
            
            # Reprojizieren
            reproject(
                source=rasterio.band(src, 1),
                destination=dem_reprojected,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=ortho_crs,
                resampling=Resampling.bilinear
            )
            
            # Auf -100 bis 10000m clippen
            dem_clipped = np.clip(dem_reprojected, min_height, max_height)
            
            # Auf 0-65535 normalisieren
            normalized = (dem_clipped - min_height) / (max_height - min_height)
            heightmap_16bit = (normalized * 65535).astype(np.uint16)
            
            # Als 16-bit TIFF speichern
            profile = {
                'driver': 'GTiff',
                'dtype': rasterio.uint16,
                'width': width,
                'height': height,
                'count': 1,
                'crs': ortho_crs,
                'transform': transform,
                'compress': 'lzw'
            }
            
            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heightmap_16bit, 1)
        
        print(f"  Konvertiert: {filename}")
    
    print(f"Fertig. Dateien in: {output_path}")
    

# Beispiel-Aufruf:
if __name__ == "__main__":
    convert_to_16bit_heightmaps()

Konvertiere 60 Dateien...
  Konvertiert: 9_2_dem_10.2515_46.3518_water.tif
Überspringe 3_0_dem_3.1497_50.3580_water.tif (existiert bereits)
  Konvertiert: 7_0_dem_-3.0215_42.6649.tif
Überspringe 3_2_dem_-0.4576_53.2139_water.tif (existiert bereits)
Überspringe 3_1_dem_-2.4971_53.1608.tif (existiert bereits)
  Konvertiert: 6_1_dem_-0.5915_48.8978.tif
Überspringe 1_1_dem_-0.0386_52.5027_water.tif (existiert bereits)
  Konvertiert: 9_1_dem_10.1733_44.0430.tif
  Konvertiert: 9_2_dem_10.2515_46.3518.tif
  Konvertiert: 9_1_dem_10.1733_44.0430_water.tif
Überspringe 3_1_dem_-2.4971_53.1608_water.tif (existiert bereits)
Überspringe 5_0_dem_8.4562_44.9047_water.tif (existiert bereits)
Überspringe 3_0_dem_3.1497_50.3580.tif (existiert bereits)
  Konvertiert: 7_0_dem_-3.0215_42.6649_water.tif
  Konvertiert: 8_0_dem_11.5893_44.0062.tif
Überspringe 4_2_dem_4.0064_43.6073_water.tif (existiert bereits)
Überspringe 0_0_dem_12.7796_45.4373.tif (existiert bereits)
  Konvertiert: 7_1_dem_-8.9244_52.0231.t